# 📦 Notebook 1 — MUCS Data Prep

**Nothing needs to be on Drive before you start. Run this once.**

What this does:
1. Mounts your Drive
2. Downloads MUCS data from OpenSLR 104 → saves to Drive
3. Parses Kaldi format transcripts (`transcripts/text`) and segment boundaries (`transcripts/segments`)
4. Filters code-switched utterances → tier split → saves CSVs with exact `start_sec` and `end_sec` timestamps to Drive

> **Why save audio to Drive?** Notebooks 2/3/4 run in separate Colab sessions. Drive is the only shared storage between them.

Dataset: MUCS 2021 Bengali-English | CS Tech Lectures | 46 hrs | CC BY-SA 4.0  
Source: `https://www.openslr.org/resources/104/` (no registration)

---
⏭️ After this runs: open Notebooks 2, 3, 4 in separate Colab tabs and train in parallel.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive + set paths
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os

# All data goes to Drive so other Colab sessions can access it
DRIVE_AUDIO = '/content/drive/MyDrive/hoichoi/mucs_raw'   # audio saved here
DRIVE_DATA  = '/content/drive/MyDrive/hoichoi/data'       # CSVs saved here

os.makedirs(DRIVE_AUDIO, exist_ok=True)
os.makedirs(DRIVE_DATA,  exist_ok=True)

print('✅ Drive mounted')
print('   Audio → ', DRIVE_AUDIO)
print('   CSVs  → ', DRIVE_DATA)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Install
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'scikit-learn'], capture_output=True)
print('✅ Done')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Download MUCS → Drive & Extract
# OpenSLR SLR104 — ~3.9 GB
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, tarfile, urllib.request
from pathlib import Path

URL      = 'https://www.openslr.org/resources/104/Bengali-English_train.tar.gz'
TAR_PATH = DRIVE_AUDIO + '/Bengali-English_train.tar.gz'
DATA_DIR = Path(DRIVE_AUDIO)

# Check if already extracted (search for Kaldi text and segments)
text_files = list(DATA_DIR.rglob('text'))
seg_files  = list(DATA_DIR.rglob('segments'))
wav_files  = list(DATA_DIR.rglob('*.wav'))

if text_files and seg_files and len(wav_files) > 10:
    print(f'✅ Already extracted on Drive ({len(wav_files)} audio recordings found).')
    print('   Skipping download & extract.')
else:
    if not os.path.exists(TAR_PATH):
        print('Downloading ~3.9 GB from OpenSLR → Drive ...')
        print('(This takes ~5-10 min. Do not close the tab.)')
        def _prog(count, block, total):
            mb  = count * block / 1e6
            tot = total / 1e6
            pct = min(100, int(mb * 100 / max(tot, 1)))
            print(f'\r  {pct}%  {mb:.0f} / {tot:.0f} MB', end='', flush=True)
        urllib.request.urlretrieve(URL, TAR_PATH, reporthook=_prog)
        print('\n✅ Download done')
    else:
        print('Archive already on Drive — skipping download')

    print('Extracting tar archive ...')
    with tarfile.open(TAR_PATH, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('✅ Extracted to', DATA_DIR)

# Confirm Kaldi files exist
text_files = list(DATA_DIR.rglob('text'))
seg_files  = list(DATA_DIR.rglob('segments'))
if not text_files or not seg_files:
    raise RuntimeError('❌ transcripts/text or transcripts/segments missing after extract. Check archive contents.')
print(f'Found {len(text_files)} text file(s) and {len(seg_files)} segments file(s).')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Parse Kaldi Metadata, Filter & Tier-Split
# audio_path + start_sec + end_sec saved so Notebooks 2/3/4 load exact slices
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import random, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TARGET = 2000
SEED   = 42
OVS    = {'high': 3, 'mid': 2, 'base': 1}
random.seed(SEED); np.random.seed(SEED)

BENGALI = (0x0980, 0x09FF)
LATIN   = (0x0041, 0x007A)

def script(c):
    cp = ord(c)
    if BENGALI[0] <= cp <= BENGALI[1]: return 'b'
    if LATIN[0]   <= cp <= LATIN[1]:   return 'l'
    return ''

def count_switches(text):
    sc = [script(c) for c in text if script(c)]
    if not sc: return 0
    n, prev = 0, sc[0]
    for s in sc[1:]:
        if s != prev: n += 1; prev = s
    return n

def tier(n):
    if n >= 3:   return 'high'  # High-density: 3+ switches (oversample 3x)
    elif n == 2: return 'mid'   # Mid-density: 2 switches (oversample 2x)
    else:        return 'base'  # Base / Low tier: 1 switch (1x weight)

# ── Parse Kaldi segments & text ───────────────────────────
DATA_DIR = Path(DRIVE_AUDIO)
seg_files  = [p for p in DATA_DIR.rglob('segments') if 'test' not in str(p).lower() and p.is_file()]
text_files = [p for p in DATA_DIR.rglob('text')     if 'test' not in str(p).lower() and p.is_file()]
wav_map    = {p.stem: str(p) for p in DATA_DIR.rglob('*.wav')}

print(f'Audio recordings found on Drive : {len(wav_map)}')

# 1. Load segments: utt_id -> (rec_id, start_sec, end_sec)
seg_dict = {}
for sf in seg_files:
    for line in open(sf, encoding='utf-8'):
        parts = line.strip().split()
        if len(parts) >= 4:
            uid, rec_id, s_sec, e_sec = parts[0], parts[1], float(parts[2]), float(parts[3])
            seg_dict[uid] = (rec_id, s_sec, e_sec)
print(f'Total segments parsed          : {len(seg_dict)}')

# 2. Load text and link to audio slice
rows = []
for tf in text_files:
    for line in open(tf, encoding='utf-8'):
        line = line.strip()
        if not line or line.startswith('#'): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        uid, raw_text = parts[0], parts[1]
        
        if uid not in seg_dict: continue
        rec_id, s_sec, e_sec = seg_dict[uid]
        apath = wav_map.get(rec_id)
        if not apath: continue
        
        trans = unicodedata.normalize('NFC', raw_text.strip())
        n = count_switches(trans)
        rows.append({
            'utt_id': uid,
            'audio_path': apath,
            'start_sec': s_sec,
            'end_sec': e_sec,
            'duration': round(e_sec - s_sec, 2),
            'transcript': trans,
            'n_switches': n,
            'tier': tier(n)
        })

print(f'Total valid audio-transcript pairs : {len(rows)}')
switched = [r for r in rows if r['n_switches'] >= 1]
print(f'Code-switched utterances           : {len(switched)} ({len(switched)*100/max(len(rows),1):.1f}%)')
if not switched:
    raise RuntimeError('❌ No code-switched utterances found. Check parsing above.')

# ── Oversample + stratified sample ───────────────────────
expanded = []
for r in switched: expanded.extend([r] * OVS[r['tier']])
random.shuffle(expanded)
df = pd.DataFrame(expanded)

fracs   = df['tier'].value_counts(normalize=True)
sampled = pd.concat([
    df[df['tier']==t].sample(
        min(max(1, round(f * TARGET)), len(df[df['tier']==t])),
        random_state=SEED)
    for t, f in fracs.items()
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df, val_df = train_test_split(
    sampled, test_size=0.1, random_state=SEED, stratify=sampled['tier'])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# ── Save CSVs to Drive ────────────────────────────────────
for t in ['high', 'mid', 'base']:
    path = DRIVE_DATA + '/train_' + t + '.csv'
    train_df[train_df['tier']==t].to_csv(path, index=False, encoding='utf-8')
    print(f'  train_{t}.csv  →  {(train_df["tier"]==t).sum()} rows')
train_df.to_csv(DRIVE_DATA + '/train_all.csv', index=False, encoding='utf-8')
val_df.to_csv(  DRIVE_DATA + '/val.csv',       index=False, encoding='utf-8')
print(f'Train total: {len(train_df)}  |  Val: {len(val_df)}')
print('✅ All CSVs saved to Drive')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Sanity check
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, pandas as pd

print('CSVs on Drive:')
for fname in ['train_high.csv', 'train_mid.csv', 'train_base.csv', 'val.csv']:
    p  = DRIVE_DATA + '/' + fname
    ok = os.path.exists(p)
    n  = len(pd.read_csv(p)) if ok else 0
    print(f'  {"✅" if ok else "❌"}  {fname:<22}  {n} rows')

# Check audio_path and timestamps
df_h   = pd.read_csv(DRIVE_DATA + '/train_high.csv')
sample = df_h.iloc[0]
print(f'\nSample record from train_high.csv:')
print(f'  utt_id     : {sample["utt_id"]}')
print(f'  audio_path : {sample["audio_path"]}')
print(f'  start_sec  : {sample["start_sec"]} s  |  end_sec: {sample["end_sec"]} s  (duration: {sample.get("duration")} s)')
print(f'  transcript : {sample["transcript"]}')
print(f'  switches   : {sample["n_switches"]} switches')

print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('✅ Notebook 1 complete!')
print('   Open Notebooks 2, 3, 4 in separate Colab tabs → run in parallel.')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')